# Cache Encoder Features (COCO 2017 → ResNet50 2048-d vectors)

One-time pre-processing: runs frozen ResNet50 over every COCO 2017 image,
saves pooled 2048-d fp16 vectors as `.npy` files (numeric 12-digit IDs).
Also builds `vocab.json` and `image_splits.json`.

**Add these Datasets before running:**
- `awsaf49/coco-2017-dataset` (or the COCO 2017 images)

**After running:** Upload `/kaggle/working/features/` as a new Kaggle Dataset.

In [ ]:
# P100 (sm_60) needs torch compiled with CUDA 11.8
!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118 -q
import torch; print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
import os, sys

REPO_URL = "https://github.com/MohamedShakshak/Image-Captioning.git"
if not os.path.isdir("Image-Captioning"):
    !git clone --depth 1 {REPO_URL}
%cd Image-Captioning

!pip install -e . -q
_src = os.path.join(os.getcwd(), "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
# SET THESE to match your Kaggle mounts
COCO_ROOT = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
ANNO_TRAIN = f"{COCO_ROOT}/annotations/captions_train2017.json"
ANNO_VAL = f"{COCO_ROOT}/annotations/captions_val2017.json"
OUTPUT_DIR = "/kaggle/working/features"

for p in [COCO_ROOT, ANNO_TRAIN, ANNO_VAL]:
    assert os.path.exists(p), f"Missing: {p}"
    print(f"OK: {p}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
!python scripts/cache_features.py \
    --coco-root {COCO_ROOT} \
    --annotation-train {ANNO_TRAIN} \
    --annotation-val {ANNO_VAL} \
    --out {OUTPUT_DIR}

In [ ]:
!python scripts/build_vocab.py \
    --annotation-train {ANNO_TRAIN} \
    --annotation-val {ANNO_VAL} \
    --out {OUTPUT_DIR}/vocab.json

In [ ]:
import os
npy_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".npy")]
print(f"{len(npy_files)} feature files")
if npy_files:
    import numpy as np
    feat = np.load(os.path.join(OUTPUT_DIR, npy_files[0]))
    print(f"Sample shape: {feat.shape} dtype: {feat.dtype}")
    print(f"First 3: {npy_files[:3]}")

In [ ]:
!du -sh {OUTPUT_DIR}

## Upload
Go to Kaggle file browser → `/kaggle/working/features/` → select all files →
**New Dataset** → name it e.g. `coco-features-2048`.

Then mount it in your training notebook and set `FEATURES_DIR` accordingly.